# Causal Inference in Practice
## Week 2 — Potential Outcomes · Practice Notebook

> **Block I — Foundations**
>
> The Neyman–Rubin framework: counterfactuals, estimands, and the assumptions that license a causal claim.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · The science table — two outcomes per unit

Causal inference would be easy if nature showed us, for every unit, **both** the treated outcome `Y(1)` and the untreated outcome `Y(0)`. That full table is the *science table*. Because we build it here in simulation, we **know the true effect for every unit** — and so the true ATE exactly. Later we'll hide one column and see how well different study designs recover that number.

In [ ]:
# Build a finite-population science table with a KNOWN true ATE.
N = 20_000
X  = RNG.normal(size=N)                      # baseline covariate (e.g. risk)
Y0 = 1.0 + 0.8 * X + RNG.normal(size=N)      # potential outcome if UNtreated
tau = 2.0 + 0.5 * X                          # heterogeneous TRUE effect
Y1 = Y0 + tau                                # potential outcome if treated

science = pd.DataFrame({'X': X, 'Y0': Y0, 'Y1': Y1, 'effect': Y1 - Y0})
TRUE_ATE = science['effect'].mean()
print(science.head())
print(f'\nTRUE ATE = E[Y(1) - Y(0)] = {TRUE_ATE:.3f}')

Every row has an individual effect `Y1 - Y0`, and the effect varies with `X` (it is *heterogeneous*). The population average of that column is the **true ATE** — about `2.0`, since `tau = 2 + 0.5*X` and `X` is mean-zero. Hold onto `TRUE_ATE`: it is the answer key for the rest of the notebook.

## 2 · The fundamental problem — you only ever see one column

Assign a treatment `A` and reveal only the matching potential outcome through the **switching equation** `Y = A·Y(1) + (1−A)·Y(0)`. The other column becomes the unobserved counterfactual. This is the data a real study would hand you.

In [ ]:
A = RNG.binomial(1, 0.5, N)                  # complete randomization, 50/50
Y = np.where(A == 1, Y1, Y0)                 # switching equation

obs = pd.DataFrame({'X': X, 'A': A, 'Y': Y})
# The counterfactual column is gone: we don't get Y0 for treated units,
# nor Y1 for untreated units.
print(obs.head())
n_missing = N   # exactly one potential outcome is hidden per unit
print(f'\nHidden counterfactuals: {n_missing} of {2*N} cells '
      f'({100*n_missing/(2*N):.0f}% of the science table).')

Half the science table is gone — one cell per row. No estimator can ever look at an individual's missing cell. The best we can do is recover a **population average**, and only when the units we see treated are a fair stand-in for the units we see untreated.

## 3 · Randomization recovers the ATE (in expectation)

Because `A` was a fair coin flip, it is independent of `{Y(0), Y(1)}`: the treated and untreated groups are *exchangeable*. So the **simple difference in observed group means** should estimate the ATE. One sample carries noise; let's check a single draw, then average over many random assignments to see it is right *in expectation*.

In [ ]:
def diff_in_means(A, Y):
    return Y[A == 1].mean() - Y[A == 0].mean()

single = diff_in_means(A, Y)
print(f'one randomized sample: {single:.3f}   (true {TRUE_ATE:.3f})')

# Average the estimator over many independent random assignments.
ests = []
for _ in range(2000):
    a = RNG.binomial(1, 0.5, N)
    y = np.where(a == 1, Y1, Y0)
    ests.append(diff_in_means(a, y))
ests = np.array(ests)
print(f'mean over 2000 assignments: {ests.mean():.3f}   (true {TRUE_ATE:.3f})')
assert abs(ests.mean() - TRUE_ATE) < 0.05, 'randomization should recover the ATE'

In [ ]:
# The sampling distribution of the randomized estimator is centred on the truth.
fig, ax = plt.subplots()
ax.hist(ests, bins=40, color='#4C72B0', alpha=0.85)
ax.axvline(TRUE_ATE, color='crimson', lw=2, label=f'true ATE = {TRUE_ATE:.2f}')
ax.axvline(ests.mean(), color='black', lw=2, ls='--',
           label=f'mean estimate = {ests.mean():.2f}')
ax.set_xlabel('difference-in-means estimate'); ax.set_ylabel('count')
ax.set_title('Randomization: unbiased for the ATE'); ax.legend()
None  # figure created; no blocking show()

The histogram of estimates is centred on the red truth line: individual experiments scatter, but the procedure is **unbiased**. That is exactly what 'recovers the ATE in expectation' means.

## 4 · A confounded observational comparison is biased

Now keep the **same science table** but let treatment depend on the covariate `X` — higher-`X` units are more likely to be treated. Because `X` also drives the potential outcomes, treatment is now correlated with `{Y(0), Y(1)}` and exchangeability fails. The naive difference in means mixes the real effect with a baseline gap.

In [ ]:
p = 1 / (1 + np.exp(-1.5 * X))               # propensity rises with X
Ac = RNG.binomial(1, p)                      # confounded assignment
Yc = np.where(Ac == 1, Y1, Y0)

naive = diff_in_means(Ac, Yc)
print(f'confounded naive diff = {naive:.3f}   (true {TRUE_ATE:.3f})')

# The bias is exactly the baseline imbalance in Y(0) between the arms:
baseline_gap = Y0[Ac == 1].mean() - Y0[Ac == 0].mean()
print(f'baseline Y(0) gap between arms = {baseline_gap:.3f}  (this IS the bias)')
assert naive - TRUE_ATE > 0.2, 'confounding should bias the naive estimate upward'

The naive number overstates the effect: treated units had higher `X`, hence higher `Y(0)` to begin with, and that head start is wrongly credited to the treatment. The printed `baseline Y(0) gap` is precisely the selection bias term `E[Y(0)|A=1] − E[Y(0)|A=0]`. Adjusting for `X` (later weeks) is how we'd close it — here we just diagnose it.

### 🔧 Exercise 4.1 — recover the ATT, then compare to the ATE

The **ATT** is the effect among the *treated* units only: `E[Y(1) − Y(0) | A = 1]`. Because we built the science table, we can compute it *exactly* — we know both columns. Using the **confounded** assignment `Ac`, fill in the `# TODO`s to compute the true ATT and compare it to the true ATE. They differ here because effects are heterogeneous and the treated have higher `X`.

The skeleton still runs (it uses `...` placeholders); replace them.

In [ ]:
# TODO: compute the TRUE ATT among the confounded-treated units (Ac == 1),
# using the full science table (we know both Y1 and Y0 here).
treated_mask = (Ac == 1)
true_ATT = ...      # TODO: mean of (Y1 - Y0) over treated_mask
# print(f'true ATT = {true_ATT:.3f}   true ATE = {TRUE_ATE:.3f}')

### ✅ Solution 4.1

In [ ]:
treated_mask = (Ac == 1)
true_ATT = (Y1 - Y0)[treated_mask].mean()
print(f'true ATT = {true_ATT:.3f}   true ATE = {TRUE_ATE:.3f}')
# Treated units have higher X, and effect tau = 2 + 0.5*X rises with X,
# so the ATT exceeds the ATE here.
assert true_ATT > TRUE_ATE, 'with positive selection on effect, ATT > ATE'
print('ATT > ATE: the treated are exactly the units with larger effects.')

## 5 · Positivity — when there is no one to compare to

Exchangeability says we *could* adjust for `X`; **positivity** says there is actually data in every stratum to adjust *with*. Here we simulate a hard violation: above a cutoff in `X`, **everyone** is treated, so no untreated comparison unit exists there. Estimation in that region is pure extrapolation.

In [ ]:
# Deterministic assignment above a cutoff: a positivity violation.
cutoff = 0.5
Av = np.where(X > cutoff, 1, RNG.binomial(1, 0.5, N))   # forced treated when X>cutoff
Yv = np.where(Av == 1, Y1, Y0)

hi = X > cutoff
p_treated_hi = Av[hi].mean()
n_untreated_hi = int((Av[hi] == 0).sum())
print(f'For X > {cutoff}:  P(treated) = {p_treated_hi:.3f},  '
      f'untreated units available = {n_untreated_hi}')
assert n_untreated_hi == 0, 'positivity is violated: no untreated units above the cutoff'
print('No untreated units exist above the cutoff -> the effect there is NOT identified.')

Within `X > 0.5` there is no untreated unit, so `E[Y(0) | X>0.5]` has nothing to estimate from. A model with an `X` term will still print an effect for that region — but it is invented by the functional form, not supported by data. The honest move is to **restrict the estimand to the overlap region**, which we do next.

### 🔧 Exercise 5.1 — restrict to the overlap region

Estimate the ATE **only where both treatments occur** — the stratum `X ≤ cutoff`, where assignment was a 50/50 coin flip and so is (locally) randomized. Fill in the `# TODO`s and check the restricted estimate recovers the *true ATE within that region*.

In [ ]:
# TODO: restrict to the overlap region X <= cutoff and estimate there.
ok = X <= cutoff
ate_overlap_true = (Y1 - Y0)[ok].mean()        # true ATE in the overlap region
est_overlap = ...   # TODO: diff_in_means on the units with `ok`, using Av and Yv
# print(f'overlap est = {est_overlap:.3f}   true (region) = {ate_overlap_true:.3f}')

### ✅ Solution 5.1

In [ ]:
ok = X <= cutoff
ate_overlap_true = (Y1 - Y0)[ok].mean()
est_overlap = diff_in_means(Av[ok], Yv[ok])
print(f'overlap est = {est_overlap:.3f}   true (region) = {ate_overlap_true:.3f}')
assert abs(est_overlap - ate_overlap_true) < 0.1, 'should recover the region ATE'
print('Honest non-identification: we report the effect only where data support it.')

## 6 · Wrap-up & self-check

- A causal effect is a contrast of **potential outcomes** `Y(1) − Y(0)`; we never see both for one unit (the **fundamental problem**).
- **State the estimand first** — ATE, ATT, or CATE. We saw ATT > ATE when the treated are selected on larger effects.
- **Randomization** makes `A ⟂ {Y(0), Y(1)}`, so the difference-in-means is unbiased — confirmed by averaging over many assignments.
- A **confounded** comparison is biased by exactly the baseline `Y(0)` gap between arms.
- **Positivity** can fail outright; the honest fix is to restrict the estimand to the overlap region.

**You're ready for Week 3** if you can write ATE/ATT/CATE from memory, name the four assumptions, and say why a coin flip buys exchangeability. Next week: designing and running randomized experiments and A/B tests well.